# Loading .tok dataset files

- Read .tok files (train, val, test).
- Choose a sequence length (512 tokens).
- Chop the dataset into sequences of length 512.
- Create batches of these sequences for training.

In [2]:
import torch
from torch.utils.data import IterableDataset, Dataset, DataLoader

Working
- Reading the .tok file and cutting them into fixed-length chunks (512 tokens)
- Splits into non-overlapping chunks of length seq_len.
- Each `__getitem__` gives a (input_ids, labels) pair.
- Labels = input_ids

In [3]:
class TokDataset(Dataset):
    def __init__(self, filepath, seq_len=512):
        # Load all token IDs from file
        with open(filepath, "r", encoding="utf-8") as f:
            data = f.read().split()
        self.tokens = list(map(int, data))  # convert to integers
        self.seq_len = seq_len

        # Number of sequences we can make
        self.num_sequences = len(self.tokens) // seq_len

    def __len__(self):
        return self.num_sequences

    def __getitem__(self, idx):
        start = idx * self.seq_len
        end = start + self.seq_len
        input_ids = torch.tensor(self.tokens[start:end], dtype=torch.long)

        # Labels are the same as inputs (next-token prediction)
        labels = input_ids.clone()
        return {"input_ids": input_ids, "labels": labels}

Working
- Reads the .tok file line by line.
- Collects tokens into a buffer of length seq_len (512).
- When full, yields a training sample {input_ids, labels}.
- hence, never loads the entire file into memory.

In [4]:
class StreamingTokDataset(IterableDataset):
    def __init__(self, filepath, seq_len=512):
        self.filepath = filepath
        self.seq_len = seq_len

    def parse_file(self):
        buffer = []
        with open(self.filepath, "r", encoding="utf-8") as f:
            for line in f:
                tokens = list(map(int, line.strip().split()))
                for tok in tokens:
                    buffer.append(tok)
                    if len(buffer) == self.seq_len:
                        input_ids = torch.tensor(buffer, dtype=torch.long)
                        labels = input_ids.clone()
                        yield {"input_ids": input_ids, "labels": labels}
                        buffer = []  # reset buffer

    def __iter__(self):
        return self.parse_file()

In [5]:
if __name__ == "__main__":
    train_dataset = StreamingTokDataset("data/multilang_train.tok", seq_len=128)
    val_dataset = TokDataset("data/multilang_val.tok", seq_len=128)
    test_dataset = TokDataset("data/multilang_test.tok", seq_len=128)

In [7]:
train_loader = DataLoader(train_dataset, batch_size=1)
val_loader = DataLoader(val_dataset, batch_size=1)
test_loader = DataLoader(test_dataset, batch_size=1)

In [8]:
for batch in train_loader:
    print(batch["input_ids"].shape)  # (batch_size, 512)
    print(batch["labels"].shape)     # (batch_size, 512)
    break

torch.Size([1, 128])
torch.Size([1, 128])


# Model Setup

In [1]:
from transformers import AutoConfig, AutoModelForCausalLM

cfg = AutoConfig.from_pretrained("google/gemma-3-1b-pt")

cfg.vocab_size = 50_000
cfg.hidden_size = 640
cfg.num_hidden_layers = 12
cfg.num_attention_heads = 10
cfg.intermediate_size = 2560
cfg.max_position_embeddings = 1024  # context length you’ll train with
cfg.tie_word_embeddings = True      # tie LM head to embeddings (saves params)

# Build the model from scratch (no pretrained weights)
model = AutoModelForCausalLM.from_config(cfg, attn_implementation="eager")

total_params = sum(p.numel() for p in model.parameters())
print(f"Total params: {total_params/1e6:.2f}M")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total params: 134.27M


Convert .tok into Hugging Face dataset (.arrow format)

In [4]:
from datasets import load_dataset

files = {
    "train": "data/multilang_train.tok",
    "validation": "data/multilang_val.tok",
    "test": "data/multilang_test.tok",
}

ds = load_dataset("text", data_files=files)
ds = ds.map(
    lambda x: {
        "input_ids": [int(i) for i in x["text"].split()],
        "labels": [int(i) for i in x["text"].split()],
    },
    remove_columns=["text"],
    num_proc=6
)

ds.save_to_disk("/Volumes/WD ExFAT/new/multilang_tok_dataset")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Saving the dataset (6/6 shards): 100%|██████████| 8764189/8764189 [00:57<00:00, 152178.37 examples/s]


# Training

In [17]:
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)

model.gradient_checkpointing_enable()

In [ ]:
from transformers import get_scheduler
from torch.optim import AdamW

# Optimizer & Scheduler
optimizer = AdamW(model.parameters(), lr=5e-5)
num_training_steps = 200
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

# AMP + Training
scaler = torch.amp.GradScaler(device="mps")

model.train()
for step, batch in enumerate(train_loader):
    inputs = {k: v.to(device) for k, v in batch.items()}

    optimizer.zero_grad()
    with torch.autocast("mps"):
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs.get("attention_mask", None),
            labels=inputs["input_ids"],   # <-- pass labels only once
        )
        loss = outputs.loss

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    lr_scheduler.step()

    if step % 20 == 0:
        print(f"Step {step} | Loss: {loss.item():.4f}")

    if step > num_training_steps:
        break

# Validation
model.eval()
val_loss, val_steps = 0, 0
with torch.no_grad():
    for step, batch in enumerate(val_loader):
        inputs = {k: v.to(device) for k, v in batch.items()}
        with torch.amp.autocast("mps"):
            outputs = model(**inputs, labels=inputs["input_ids"])
            val_loss += outputs.loss.item()
        val_steps += 1
        if step > 50:
            break

print(f"Validation Loss (sample): {val_loss / val_steps:.4f}")

It is strongly recommended to train Gemma3 models with the `eager` attention implementation instead of `sdpa`. Use `eager` with `AutoModelForCausalLM.from_pretrained('<path-to-checkpoint>', attn_implementation='eager')`.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step 0 | Loss: 12.8366
Step 20 | Loss: 12.8451
Step 40 | Loss: 12.8297
Step 60 | Loss: 12.7944
Step 80 | Loss: 12.8664
Step 100 | Loss: 12.7494
Step 120 | Loss: 12.8174
Step 140 | Loss: 12.8445
Step 160 | Loss: 12.8082
Step 180 | Loss: 12.7937
Step 200 | Loss: 12.7914


TypeError: Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 384, padding_idx=0)
    (layers): ModuleList(
      (0-7): 8 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=384, out_features=1536, bias=False)
          (k_proj): Linear(in_features=384, out_features=256, bias=False)
          (v_proj): Linear(in_features=384, out_features=256, bias=False)
          (o_proj): Linear(in_features=1536, out_features=384, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=384, out_features=1536, bias=False)
          (up_proj): Linear(in_features=384, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=384, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((3

# Small model (Concept Proof)

In [ ]:
import os
import glob
import time
import torch
from torch.utils.data import IterableDataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoConfig, AutoModelForCausalLM
from torch.optim import AdamW
import matplotlib.pyplot as plt
train_losses = []
val_steps = []
val_losses = []
steps_recorded = []

# USER CONFIG
TRAIN_TOK_FILES = ["./data/multilang_train.tok"]
VAL_TOK_FILES   = ["./data/multilang_val.tok"]
OUTPUT_DIR      = "./data/gemma20m_ckpts"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CFG_OVERRIDE = dict(
    vocab_size=50_000,
    hidden_size=256,
    num_hidden_layers=4,
    num_attention_heads=4,
    intermediate_size=1024,
    max_position_embeddings=512,
    tie_word_embeddings=True,
)

SEQ_LEN = 512
BATCH_SIZE = 4
GRAD_ACCUM = 2
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
SAVE_EVERY_STEPS = 5000
LOG_EVERY_STEPS = 200
VAL_EVERY_STEPS = 1000
TOTAL_TOKENS_TARGET = 400_000_000

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print("DEVICE:", DEVICE)
print("OUTPUT_DIR:", OUTPUT_DIR)

# DATASET
class TokLineChunker(IterableDataset):
    def __init__(self, file_list, seq_len):
        self.file_list = file_list
        self.seq_len = seq_len

    def _line_generator(self):
        for fp in self.file_list:
            with open(os.path.expanduser(fp), "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        try:
                            yield [int(x) for x in line.split()]
                        except:
                            continue

    def __iter__(self):
        buff = []
        for toks in self._line_generator():
            buff.extend(toks)
            while len(buff) >= self.seq_len:
                block = buff[:self.seq_len]
                buff = buff[self.seq_len:]
                yield {"input_ids": torch.tensor(block, dtype=torch.long)}
        return

def collate_fn(batch):
    input_ids = [b["input_ids"] for b in batch]
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=0)
    attention_mask = (input_ids != 0).long()
    return {"input_ids": input_ids, "attention_mask": attention_mask}

train_ds = TokLineChunker(TRAIN_TOK_FILES, SEQ_LEN)
val_ds   = TokLineChunker(VAL_TOK_FILES, SEQ_LEN) if VAL_TOK_FILES else None

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, collate_fn=collate_fn, num_workers=0) if val_ds else None

# MODEL
MODEL_SOURCE = "google/gemma-3-270m"
cfg = AutoConfig.from_pretrained(MODEL_SOURCE, trust_remote_code=True)
for k, v in CFG_OVERRIDE.items():
    setattr(cfg, k, v)
cfg.max_position_embeddings = max(cfg.max_position_embeddings, SEQ_LEN)

cfg.pad_token_id = 0

model = AutoModelForCausalLM.from_config(cfg, trust_remote_code=True)
print(f"Model parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f} M")

device = torch.device(DEVICE)
model.to(device)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Linear warmup + decay
from transformers import get_scheduler
effective_tokens_per_step = BATCH_SIZE * SEQ_LEN * GRAD_ACCUM
estimated_steps = max(1, TOTAL_TOKENS_TARGET // effective_tokens_per_step)
num_warmup = max(1, int(0.03 * estimated_steps))
scheduler = get_scheduler("cosine", optimizer=optimizer, num_warmup_steps=num_warmup, num_training_steps=estimated_steps)

# RESUME CHECKPOINT
def find_latest_checkpoint(dirpath):
    ckpts = sorted(glob.glob(os.path.join(dirpath, "ckpt-*.pt")), key=os.path.getmtime)
    return ckpts[-1] if ckpts else None

start_step = 0
latest = find_latest_checkpoint(OUTPUT_DIR)
if latest:
    print("Found checkpoint:", latest, " -> loading ...")
    d = torch.load(latest, map_location=device)
    model.load_state_dict(d["model"])
    optimizer.load_state_dict(d["optim"])
    saved_cfg = d.get("config")
    if saved_cfg:
        print("Loaded saved config from checkpoint (you might want to compare it to current CFG).")
    start_step = d.get("step", 0) + 1
    print("Resumed at step:", start_step)
else:
    print("No checkpoint found. Starting fresh.")

# TRAIN LOOP
model.train()
global_step = start_step
accum_count = 0
running_loss = 0.0
log_time = time.time()
train_start_time = time.time()

try:
    while global_step < estimated_steps:
        for batch in train_loader:
            inputs = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            # Mask out padding tokens for loss (important!)
            labels = inputs.clone()
            labels[labels == cfg.pad_token_id] = -100

            outputs = model(inputs, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss_val = loss.item()

            # gradient accumulation
            loss = loss / GRAD_ACCUM
            loss.backward()
            accum_count += 1

            if accum_count >= GRAD_ACCUM:
                # gradient clipping
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                accum_count = 0
                global_step += 1
                running_loss += loss_val

                # logging (compute elapsed BEFORE resetting log_time)
                if global_step % LOG_EVERY_STEPS == 0:
                    avg_loss = running_loss / LOG_EVERY_STEPS
                    elapsed = time.time() - log_time
                    total_elapsed = (time.time() - train_start_time) / 60  # minutes
                    lr_now = scheduler.get_last_lr()[0] if hasattr(scheduler, "get_last_lr") else None
                    train_losses.append(avg_loss)
                    steps_recorded.append(global_step)
                    print(f"[step {global_step}] avg_loss={avg_loss:.4f}; lr={lr_now:.3e}; interval_elapsed={elapsed:.1f}s; total_time={total_elapsed:.1f}m")
                    running_loss = 0.0
                    log_time = time.time()

                # checkpoint
                if global_step % SAVE_EVERY_STEPS == 0:
                    ckpt_path = os.path.join(OUTPUT_DIR, f"ckpt-{global_step}.pt")
                    torch.save({
                        "model": model.state_dict(),
                        "optim": optimizer.state_dict(),
                        "step": global_step,
                        "config": cfg.to_dict()
                    }, ckpt_path)
                    print("Saved checkpoint:", ckpt_path)
                
                # validation
                if val_loader is not None and VAL_EVERY_STEPS > 0 and global_step % VAL_EVERY_STEPS == 0:
                    model.eval()
                    val_loss = 0.0
                    val_batches = 0
                    with torch.no_grad():
                        for batch in val_loader:
                            inputs_v = batch["input_ids"].to(device)
                            attention_mask_v = batch["attention_mask"].to(device)
                            labels_v = inputs_v.clone()
                            labels_v[labels_v == cfg.pad_token_id] = -100
                            outputs_v = model(inputs_v, attention_mask=attention_mask_v, labels=labels_v)
                            val_loss += outputs_v.loss.item()
                            val_batches += 1
                            if val_batches >= 50:
                                break
                    avg_val_loss = val_loss / max(1, val_batches)
                    val_steps.append(global_step)
                    val_losses.append(avg_val_loss)
                    print(f"[validation @ step {global_step}] val_loss={avg_val_loss:.4f}")
                    model.train()

                if global_step >= estimated_steps:
                    break
        if global_step >= estimated_steps:
            break

except KeyboardInterrupt:
    print("Interrupted by user. Saving checkpoint...")
    ckpt_path = os.path.join(OUTPUT_DIR, f"ckpt-{global_step}.pt")
    torch.save({
        "model": model.state_dict(),
        "optim": optimizer.state_dict(),
        "step": global_step,
        "config": cfg.to_dict()
    }, ckpt_path)
    print("Saved checkpoint:", ckpt_path)

# final save
final_path = os.path.join(OUTPUT_DIR, "final.pt")
torch.save({
    "model": model.state_dict(),
    "optim": optimizer.state_dict(),
    "step": global_step,
    "config": cfg.to_dict()
}, final_path)
print("Training finished. Final model saved to", final_path)

- Chinchilla scaling: For ~20 tokens per parameter → ~370M tokens.

#### Initial Output

```
DEVICE: mps
OUTPUT_DIR: ./data/gemma20m_ckpts
Model parameters: 18.57 M
No checkpoint found. Starting fresh.
[step 200] avg_loss=9.6581; lr=1.093e-04; interval_elapsed=228.0s; total_time=3.8m
[step 400] avg_loss=7.2038; lr=1.994e-04; interval_elapsed=217.8s; total_time=7.4m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-500.pt
[step 600] avg_loss=7.2279; lr=1.960e-04; interval_elapsed=230.4s; total_time=11.3m
[step 800] avg_loss=6.5454; lr=1.927e-04; interval_elapsed=246.1s; total_time=15.4m
[step 1000] avg_loss=6.2079; lr=1.893e-04; interval_elapsed=237.3s; total_time=19.3m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-1000.pt
[step 1200] avg_loss=6.0737; lr=1.859e-04; interval_elapsed=277.8s; total_time=24.0m
[step 1400] avg_loss=6.6349; lr=1.825e-04; interval_elapsed=239.6s; total_time=28.0m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-1500.pt
[step 1600] avg_loss=6.0231; lr=1.792e-04; interval_elapsed=275.7s; total_time=32.5m
[step 1800] avg_loss=6.2150; lr=1.758e-04; interval_elapsed=255.0s; total_time=36.8m
[step 2000] avg_loss=6.3031; lr=1.724e-04; interval_elapsed=241.0s; total_time=40.8m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-2000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x123a8c040>
Traceback (most recent call last):
  File "/var/folders/yk/ckf_sl3n48b69lg7d7k95qs00000gn/T/ipykernel_1378/2697174097.py", line 208, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 2000] val_loss=6.2741
[step 2200] avg_loss=6.1944; lr=1.690e-04; interval_elapsed=243.0s; total_time=44.9m
[step 2400] avg_loss=6.1125; lr=1.656e-04; interval_elapsed=228.7s; total_time=48.7m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-2500.pt
[step 2600] avg_loss=6.0646; lr=1.623e-04; interval_elapsed=228.7s; total_time=52.5m
[step 2800] avg_loss=5.9967; lr=1.589e-04; interval_elapsed=230.6s; total_time=56.3m
[step 3000] avg_loss=5.9619; lr=1.555e-04; interval_elapsed=227.8s; total_time=60.1m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-3000.pt
[step 3200] avg_loss=5.9346; lr=1.521e-04; interval_elapsed=228.2s; total_time=63.9m
[step 3400] avg_loss=5.8078; lr=1.488e-04; interval_elapsed=225.5s; total_time=67.7m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-3500.pt
[step 3600] avg_loss=5.7075; lr=1.454e-04; interval_elapsed=225.3s; total_time=71.4m
[step 3800] avg_loss=5.6560; lr=1.420e-04; interval_elapsed=225.0s; total_time=75.2m
[step 4000] avg_loss=5.8199; lr=1.386e-04; interval_elapsed=224.9s; total_time=78.9m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-4000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x163d63340>
Traceback (most recent call last):
  File "/var/folders/yk/ckf_sl3n48b69lg7d7k95qs00000gn/T/ipykernel_1378/2697174097.py", line 208, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 4000] val_loss=5.9247
[step 4200] avg_loss=5.8819; lr=1.352e-04; interval_elapsed=236.2s; total_time=82.9m
[step 4400] avg_loss=5.8785; lr=1.319e-04; interval_elapsed=229.1s; total_time=86.7m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-4500.pt
[step 4600] avg_loss=5.8545; lr=1.285e-04; interval_elapsed=224.4s; total_time=90.4m
[step 4800] avg_loss=5.8262; lr=1.251e-04; interval_elapsed=221.4s; total_time=94.1m
[step 5000] avg_loss=5.6977; lr=1.217e-04; interval_elapsed=233.2s; total_time=98.0m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-5000.pt
[step 5200] avg_loss=5.7174; lr=1.184e-04; interval_elapsed=222.1s; total_time=101.7m
[step 5400] avg_loss=5.7758; lr=1.150e-04; interval_elapsed=219.1s; total_time=105.4m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-5500.pt
[step 5600] avg_loss=5.5916; lr=1.116e-04; interval_elapsed=222.2s; total_time=109.1m
[step 5800] avg_loss=5.8159; lr=1.082e-04; interval_elapsed=299.0s; total_time=114.1m
[step 6000] avg_loss=5.8055; lr=1.048e-04; interval_elapsed=225.4s; total_time=117.8m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-6000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x163d61cf0>
Traceback (most recent call last):
  File "/var/folders/yk/ckf_sl3n48b69lg7d7k95qs00000gn/T/ipykernel_1378/2697174097.py", line 208, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 6000] val_loss=5.8039
[step 6200] avg_loss=5.7948; lr=1.015e-04; interval_elapsed=228.3s; total_time=121.6m
[step 6400] avg_loss=5.7665; lr=9.808e-05; interval_elapsed=218.1s; total_time=125.3m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-6500.pt
[step 6600] avg_loss=5.6009; lr=9.470e-05; interval_elapsed=217.4s; total_time=128.9m
[step 6800] avg_loss=5.7753; lr=9.133e-05; interval_elapsed=217.8s; total_time=132.5m
[step 7000] avg_loss=5.7657; lr=8.795e-05; interval_elapsed=217.5s; total_time=136.1m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-7000.pt
[step 7200] avg_loss=5.7730; lr=8.457e-05; interval_elapsed=223.9s; total_time=139.9m
[step 7400] avg_loss=5.7595; lr=8.119e-05; interval_elapsed=237.3s; total_time=143.8m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-7500.pt
[step 7600] avg_loss=5.7654; lr=7.781e-05; interval_elapsed=281.0s; total_time=148.5m
[step 7800] avg_loss=5.7607; lr=7.444e-05; interval_elapsed=251.2s; total_time=152.7m
[step 8000] avg_loss=5.7346; lr=7.106e-05; interval_elapsed=244.5s; total_time=156.8m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-8000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x123a8c040>
Traceback (most recent call last):
  File "/var/folders/yk/ckf_sl3n48b69lg7d7k95qs00000gn/T/ipykernel_1378/2697174097.py", line 208, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 8000] val_loss=5.7591
[step 8200] avg_loss=5.6769; lr=6.768e-05; interval_elapsed=241.3s; total_time=160.8m
[step 8400] avg_loss=5.5718; lr=6.430e-05; interval_elapsed=228.8s; total_time=164.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-8500.pt
[step 8600] avg_loss=5.5349; lr=6.092e-05; interval_elapsed=275.7s; total_time=169.2m
Exception ignored in: <generator object TokLineChunker._line_generator at 0x163d61f10>
Traceback (most recent call last):
  File "/var/folders/yk/ckf_sl3n48b69lg7d7k95qs00000gn/T/ipykernel_1378/2697174097.py", line 146, in <module>
RuntimeError: generator ignored GeneratorExit
Interrupted by user. Saving checkpoint...
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-8632.pt
Training finished. Final model saved to ./data/gemma20m_ckpts/final.pt
```

```
DEVICE: mps
OUTPUT_DIR: ./data/gemma20m_ckpts
Model parameters: 18.57 M
Found checkpoint: ./data/gemma20m_ckpts/ckpt-8567.pt  -> loading ...
Loaded saved config from checkpoint (you might want to compare it to current CFG).
Resumed at step: 8568
It is strongly recommended to train Gemma3 models with the `eager` attention implementation instead of `sdpa`. Use `eager` with `AutoModelForCausalLM.from_pretrained('<path-to-checkpoint>', attn_implementation='eager')`.
[step 8600] avg_loss=0.9380; lr=2.185e-06; interval_elapsed=43.4s; total_time=0.7m
[step 8800] avg_loss=5.8011; lr=1.584e-05; interval_elapsed=219.9s; total_time=4.4m
[step 9000] avg_loss=5.7597; lr=2.950e-05; interval_elapsed=222.4s; total_time=8.1m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-9000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 9000] val_loss=5.7579
[step 9200] avg_loss=5.5957; lr=4.315e-05; interval_elapsed=244.5s; total_time=12.2m
[step 9400] avg_loss=5.4944; lr=5.681e-05; interval_elapsed=234.0s; total_time=16.1m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-9500.pt
[step 9600] avg_loss=5.5031; lr=7.047e-05; interval_elapsed=236.1s; total_time=20.0m
[step 9800] avg_loss=5.5450; lr=8.412e-05; interval_elapsed=236.0s; total_time=23.9m
[step 10000] avg_loss=5.7538; lr=9.778e-05; interval_elapsed=236.7s; total_time=27.9m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-10000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 10000] val_loss=5.7627
[step 10200] avg_loss=5.4754; lr=1.114e-04; interval_elapsed=268.2s; total_time=32.4m
[step 10400] avg_loss=5.6789; lr=1.251e-04; interval_elapsed=304.0s; total_time=37.4m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-10500.pt
[step 10600] avg_loss=5.7401; lr=1.388e-04; interval_elapsed=299.0s; total_time=42.4m
[step 10800] avg_loss=5.7175; lr=1.524e-04; interval_elapsed=250.5s; total_time=46.6m
[step 11000] avg_loss=5.7103; lr=1.661e-04; interval_elapsed=243.6s; total_time=50.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-11000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 11000] val_loss=5.7537
[step 11200] avg_loss=5.7214; lr=1.797e-04; interval_elapsed=256.8s; total_time=54.9m
[step 11400] avg_loss=5.6973; lr=1.934e-04; interval_elapsed=241.7s; total_time=58.9m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-11500.pt
[step 11600] avg_loss=5.6995; lr=2.000e-04; interval_elapsed=254.4s; total_time=63.2m
[step 11800] avg_loss=5.6450; lr=2.000e-04; interval_elapsed=252.2s; total_time=67.4m
[step 12000] avg_loss=5.4983; lr=2.000e-04; interval_elapsed=293.4s; total_time=72.3m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-12000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 12000] val_loss=5.9486
[step 12200] avg_loss=5.4596; lr=2.000e-04; interval_elapsed=289.3s; total_time=77.1m
[step 12400] avg_loss=5.4589; lr=2.000e-04; interval_elapsed=272.4s; total_time=81.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-12500.pt
[step 12600] avg_loss=5.6386; lr=1.999e-04; interval_elapsed=269.6s; total_time=86.1m
[step 12800] avg_loss=5.6792; lr=1.999e-04; interval_elapsed=266.4s; total_time=90.6m
[step 13000] avg_loss=5.6902; lr=1.999e-04; interval_elapsed=283.2s; total_time=95.3m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-13000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 13000] val_loss=5.7009
[step 13200] avg_loss=5.6678; lr=1.998e-04; interval_elapsed=295.1s; total_time=100.2m
[step 13400] avg_loss=5.6497; lr=1.998e-04; interval_elapsed=298.8s; total_time=105.2m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-13500.pt
[step 13600] avg_loss=5.5012; lr=1.998e-04; interval_elapsed=263.9s; total_time=109.6m
[step 13800] avg_loss=5.5879; lr=1.997e-04; interval_elapsed=256.4s; total_time=113.9m
[step 14000] avg_loss=5.5735; lr=1.997e-04; interval_elapsed=247.0s; total_time=118.0m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-14000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 14000] val_loss=5.9153
[step 14200] avg_loss=5.4715; lr=1.996e-04; interval_elapsed=254.8s; total_time=122.2m
[step 14400] avg_loss=5.6549; lr=1.995e-04; interval_elapsed=244.4s; total_time=126.3m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-14500.pt
[step 14600] avg_loss=5.6505; lr=1.995e-04; interval_elapsed=252.9s; total_time=130.5m
[step 14800] avg_loss=5.6390; lr=1.994e-04; interval_elapsed=255.1s; total_time=134.8m
[step 15000] avg_loss=5.5846; lr=1.993e-04; interval_elapsed=244.5s; total_time=138.8m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-15000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 15000] val_loss=5.8368
[step 15200] avg_loss=5.4778; lr=1.992e-04; interval_elapsed=254.8s; total_time=143.1m
[step 15400] avg_loss=5.6151; lr=1.992e-04; interval_elapsed=239.2s; total_time=147.1m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-15500.pt
[step 15600] avg_loss=5.6126; lr=1.991e-04; interval_elapsed=236.5s; total_time=151.0m
[step 15800] avg_loss=5.6133; lr=1.990e-04; interval_elapsed=247.3s; total_time=155.1m
[step 16000] avg_loss=5.5961; lr=1.989e-04; interval_elapsed=236.0s; total_time=159.1m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-16000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 16000] val_loss=5.6129
[step 16200] avg_loss=5.6045; lr=1.988e-04; interval_elapsed=240.5s; total_time=163.1m
[step 16400] avg_loss=5.6002; lr=1.987e-04; interval_elapsed=231.2s; total_time=166.9m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-16500.pt
[step 16600] avg_loss=5.5620; lr=1.986e-04; interval_elapsed=229.9s; total_time=170.8m
[step 16800] avg_loss=5.5104; lr=1.985e-04; interval_elapsed=230.1s; total_time=174.6m
[step 17000] avg_loss=5.4269; lr=1.983e-04; interval_elapsed=230.5s; total_time=178.4m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-17000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 17000] val_loss=5.8323
[step 17200] avg_loss=5.4278; lr=1.982e-04; interval_elapsed=239.5s; total_time=182.4m
[step 17400] avg_loss=5.5690; lr=1.981e-04; interval_elapsed=229.1s; total_time=186.2m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-17500.pt
[step 17600] avg_loss=5.4046; lr=1.980e-04; interval_elapsed=228.7s; total_time=190.1m
[step 17800] avg_loss=5.5446; lr=1.978e-04; interval_elapsed=229.0s; total_time=193.9m
[step 18000] avg_loss=5.4885; lr=1.977e-04; interval_elapsed=231.0s; total_time=197.7m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-18000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 18000] val_loss=5.7883
[step 18200] avg_loss=5.3765; lr=1.975e-04; interval_elapsed=239.9s; total_time=201.7m
[step 18400] avg_loss=5.4102; lr=1.974e-04; interval_elapsed=228.2s; total_time=205.5m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-18500.pt
[step 18600] avg_loss=5.5781; lr=1.972e-04; interval_elapsed=230.2s; total_time=209.4m
[step 18800] avg_loss=5.3799; lr=1.971e-04; interval_elapsed=232.6s; total_time=213.2m
[step 19000] avg_loss=5.5157; lr=1.969e-04; interval_elapsed=228.6s; total_time=217.1m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-19000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 19000] val_loss=5.5817
[step 19200] avg_loss=5.5569; lr=1.968e-04; interval_elapsed=239.0s; total_time=221.0m
[step 19400] avg_loss=5.5422; lr=1.966e-04; interval_elapsed=230.4s; total_time=224.9m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-19500.pt
[step 19600] avg_loss=5.3714; lr=1.964e-04; interval_elapsed=230.9s; total_time=228.7m
[step 19800] avg_loss=5.3120; lr=1.962e-04; interval_elapsed=228.7s; total_time=232.5m
[step 20000] avg_loss=5.5250; lr=1.961e-04; interval_elapsed=228.8s; total_time=236.3m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-20000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 20000] val_loss=5.5717
[step 20200] avg_loss=5.4361; lr=1.959e-04; interval_elapsed=237.8s; total_time=240.3m
[step 20400] avg_loss=5.4117; lr=1.957e-04; interval_elapsed=227.6s; total_time=244.1m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-20500.pt
[step 20600] avg_loss=5.5679; lr=1.955e-04; interval_elapsed=226.7s; total_time=247.9m
[step 20800] avg_loss=5.5490; lr=1.953e-04; interval_elapsed=226.2s; total_time=251.7m
[step 21000] avg_loss=5.3651; lr=1.951e-04; interval_elapsed=226.9s; total_time=255.4m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-21000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 21000] val_loss=5.6938
[step 21200] avg_loss=5.3098; lr=1.949e-04; interval_elapsed=238.4s; total_time=259.4m
[step 21400] avg_loss=5.4496; lr=1.947e-04; interval_elapsed=235.6s; total_time=263.3m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-21500.pt
[step 21600] avg_loss=5.5561; lr=1.944e-04; interval_elapsed=236.8s; total_time=267.3m
[step 21800] avg_loss=5.5280; lr=1.942e-04; interval_elapsed=239.2s; total_time=271.3m
[step 22000] avg_loss=5.2850; lr=1.940e-04; interval_elapsed=240.3s; total_time=275.3m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-22000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 22000] val_loss=5.6990
[step 22200] avg_loss=5.2927; lr=1.938e-04; interval_elapsed=246.2s; total_time=279.4m
[step 22400] avg_loss=5.4582; lr=1.935e-04; interval_elapsed=242.1s; total_time=283.4m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-22500.pt
[step 22600] avg_loss=5.5253; lr=1.933e-04; interval_elapsed=225.5s; total_time=287.2m
[step 22800] avg_loss=5.5230; lr=1.931e-04; interval_elapsed=228.1s; total_time=291.0m
[step 23000] avg_loss=5.5216; lr=1.928e-04; interval_elapsed=235.9s; total_time=294.9m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-23000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 23000] val_loss=5.5340
[step 23200] avg_loss=5.4950; lr=1.926e-04; interval_elapsed=236.3s; total_time=298.8m
[step 23400] avg_loss=5.5113; lr=1.923e-04; interval_elapsed=231.3s; total_time=302.7m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-23500.pt
[step 23600] avg_loss=5.5028; lr=1.921e-04; interval_elapsed=228.9s; total_time=306.5m
[step 23800] avg_loss=5.4874; lr=1.918e-04; interval_elapsed=235.6s; total_time=310.4m
[step 24000] avg_loss=5.3254; lr=1.915e-04; interval_elapsed=220.5s; total_time=314.1m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-24000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 24000] val_loss=5.6356
[step 24200] avg_loss=5.5109; lr=1.913e-04; interval_elapsed=228.3s; total_time=317.9m
[step 24400] avg_loss=5.3309; lr=1.910e-04; interval_elapsed=218.5s; total_time=321.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-24500.pt
[step 24600] avg_loss=5.2838; lr=1.907e-04; interval_elapsed=218.8s; total_time=325.2m
[step 24800] avg_loss=5.2884; lr=1.904e-04; interval_elapsed=218.5s; total_time=328.9m
[step 25000] avg_loss=5.3573; lr=1.901e-04; interval_elapsed=218.7s; total_time=332.5m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-25000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 25000] val_loss=5.5233
[step 25200] avg_loss=5.5125; lr=1.899e-04; interval_elapsed=228.9s; total_time=336.3m
[step 25400] avg_loss=5.4861; lr=1.896e-04; interval_elapsed=218.5s; total_time=340.0m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-25500.pt
[step 25600] avg_loss=5.5000; lr=1.893e-04; interval_elapsed=218.8s; total_time=343.6m
[step 25800] avg_loss=5.3824; lr=1.890e-04; interval_elapsed=218.5s; total_time=347.2m
[step 26000] avg_loss=5.3539; lr=1.887e-04; interval_elapsed=218.6s; total_time=350.9m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-26000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 26000] val_loss=5.5115
[step 26200] avg_loss=5.4710; lr=1.883e-04; interval_elapsed=228.4s; total_time=354.7m
[step 26400] avg_loss=5.2434; lr=1.880e-04; interval_elapsed=218.6s; total_time=358.3m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-26500.pt
[step 26600] avg_loss=5.4809; lr=1.877e-04; interval_elapsed=218.8s; total_time=362.0m
[step 26800] avg_loss=5.3475; lr=1.874e-04; interval_elapsed=218.6s; total_time=365.6m
[step 27000] avg_loss=5.3561; lr=1.871e-04; interval_elapsed=218.5s; total_time=369.3m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-27000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 27000] val_loss=5.5045
[step 27200] avg_loss=5.4770; lr=1.867e-04; interval_elapsed=228.3s; total_time=373.1m
[step 27400] avg_loss=5.4874; lr=1.864e-04; interval_elapsed=218.5s; total_time=376.7m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-27500.pt
[step 27600] avg_loss=5.2624; lr=1.861e-04; interval_elapsed=218.7s; total_time=380.4m
[step 27800] avg_loss=5.2248; lr=1.857e-04; interval_elapsed=218.5s; total_time=384.0m
[step 28000] avg_loss=5.3948; lr=1.854e-04; interval_elapsed=218.5s; total_time=387.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-28000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 28000] val_loss=5.5009
[step 28200] avg_loss=5.4083; lr=1.850e-04; interval_elapsed=228.4s; total_time=391.5m
[step 28400] avg_loss=5.7293; lr=1.847e-04; interval_elapsed=218.6s; total_time=395.1m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-28500.pt
[step 28600] avg_loss=7.8435; lr=1.843e-04; interval_elapsed=218.8s; total_time=398.7m
[step 28800] avg_loss=7.0909; lr=1.840e-04; interval_elapsed=218.5s; total_time=402.4m
[step 29000] avg_loss=6.8526; lr=1.836e-04; interval_elapsed=218.5s; total_time=406.0m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-29000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 29000] val_loss=5.9870
[step 29200] avg_loss=5.6565; lr=1.833e-04; interval_elapsed=228.2s; total_time=409.8m
[step 29400] avg_loss=5.4944; lr=1.829e-04; interval_elapsed=218.5s; total_time=413.5m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-29500.pt
[step 29600] avg_loss=5.4182; lr=1.825e-04; interval_elapsed=219.1s; total_time=417.1m
[step 29800] avg_loss=5.2560; lr=1.821e-04; interval_elapsed=218.5s; total_time=420.8m
[step 30000] avg_loss=5.2530; lr=1.818e-04; interval_elapsed=218.4s; total_time=424.4m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-30000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 30000] val_loss=5.5341
[step 30200] avg_loss=5.4863; lr=1.814e-04; interval_elapsed=228.4s; total_time=428.2m
[step 30400] avg_loss=5.2870; lr=1.810e-04; interval_elapsed=219.0s; total_time=431.9m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-30500.pt
[step 30600] avg_loss=5.2382; lr=1.806e-04; interval_elapsed=218.9s; total_time=435.5m
[step 30800] avg_loss=5.2290; lr=1.802e-04; interval_elapsed=218.9s; total_time=439.2m
[step 31000] avg_loss=5.2396; lr=1.798e-04; interval_elapsed=218.6s; total_time=442.8m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-31000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 31000] val_loss=5.6503
[step 31200] avg_loss=5.2749; lr=1.794e-04; interval_elapsed=228.3s; total_time=446.6m
[step 31400] avg_loss=5.4961; lr=1.790e-04; interval_elapsed=218.5s; total_time=450.2m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-31500.pt
[step 31600] avg_loss=5.2475; lr=1.786e-04; interval_elapsed=219.5s; total_time=453.9m
[step 31800] avg_loss=5.4060; lr=1.782e-04; interval_elapsed=218.5s; total_time=457.5m
[step 32000] avg_loss=5.3743; lr=1.778e-04; interval_elapsed=218.9s; total_time=461.2m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-32000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 32000] val_loss=5.5665
[step 32200] avg_loss=5.2792; lr=1.773e-04; interval_elapsed=228.3s; total_time=465.0m
[step 32400] avg_loss=5.4818; lr=1.769e-04; interval_elapsed=218.4s; total_time=468.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-32500.pt
[step 32600] avg_loss=5.2478; lr=1.765e-04; interval_elapsed=218.7s; total_time=472.3m
[step 32800] avg_loss=5.4121; lr=1.761e-04; interval_elapsed=218.5s; total_time=475.9m
[step 33000] avg_loss=5.3461; lr=1.756e-04; interval_elapsed=218.5s; total_time=479.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-33000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 33000] val_loss=5.5560
[step 33200] avg_loss=5.7328; lr=1.752e-04; interval_elapsed=228.4s; total_time=483.4m
[step 33400] avg_loss=6.7117; lr=1.748e-04; interval_elapsed=218.6s; total_time=487.0m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-33500.pt
[step 33600] avg_loss=6.6356; lr=1.743e-04; interval_elapsed=218.8s; total_time=490.7m
[step 33800] avg_loss=6.5355; lr=1.739e-04; interval_elapsed=218.5s; total_time=494.3m
[step 34000] avg_loss=5.4816; lr=1.734e-04; interval_elapsed=219.5s; total_time=498.0m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-34000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 34000] val_loss=5.4930
[step 34200] avg_loss=5.4894; lr=1.730e-04; interval_elapsed=228.5s; total_time=501.8m
[step 34400] avg_loss=5.4843; lr=1.725e-04; interval_elapsed=219.0s; total_time=505.4m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-34500.pt
[step 34600] avg_loss=5.4223; lr=1.721e-04; interval_elapsed=219.5s; total_time=509.1m
[step 34800] avg_loss=5.1928; lr=1.716e-04; interval_elapsed=218.8s; total_time=512.7m
[step 35000] avg_loss=5.4777; lr=1.711e-04; interval_elapsed=219.1s; total_time=516.4m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-35000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 35000] val_loss=5.4825
[step 35200] avg_loss=5.4634; lr=1.707e-04; interval_elapsed=228.4s; total_time=520.2m
[step 35400] avg_loss=5.9758; lr=1.702e-04; interval_elapsed=219.1s; total_time=523.8m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-35500.pt
[step 35600] avg_loss=6.5294; lr=1.697e-04; interval_elapsed=218.8s; total_time=527.5m
[step 35800] avg_loss=6.4832; lr=1.692e-04; interval_elapsed=218.6s; total_time=531.1m
[step 36000] avg_loss=6.3120; lr=1.688e-04; interval_elapsed=218.6s; total_time=534.8m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-36000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 36000] val_loss=5.4890
[step 36200] avg_loss=5.4655; lr=1.683e-04; interval_elapsed=228.4s; total_time=538.6m
[step 36400] avg_loss=5.4742; lr=1.678e-04; interval_elapsed=218.5s; total_time=542.2m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-36500.pt
[step 36600] avg_loss=5.3366; lr=1.673e-04; interval_elapsed=218.8s; total_time=545.9m
[step 36800] avg_loss=5.3254; lr=1.668e-04; interval_elapsed=218.5s; total_time=549.5m
[step 37000] avg_loss=5.4099; lr=1.663e-04; interval_elapsed=218.5s; total_time=553.1m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-37000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 37000] val_loss=5.5398
[step 37200] avg_loss=5.2177; lr=1.658e-04; interval_elapsed=228.3s; total_time=557.0m
[step 37400] avg_loss=5.2040; lr=1.653e-04; interval_elapsed=218.4s; total_time=560.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-37500.pt
[step 37600] avg_loss=5.4103; lr=1.648e-04; interval_elapsed=218.8s; total_time=564.2m
[step 37800] avg_loss=5.4638; lr=1.643e-04; interval_elapsed=218.5s; total_time=567.9m
[step 38000] avg_loss=5.7703; lr=1.638e-04; interval_elapsed=218.5s; total_time=571.5m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-38000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 38000] val_loss=5.5805
[step 38200] avg_loss=6.4348; lr=1.633e-04; interval_elapsed=228.4s; total_time=575.3m
[step 38400] avg_loss=6.4157; lr=1.628e-04; interval_elapsed=218.9s; total_time=579.0m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-38500.pt
[step 38600] avg_loss=6.3862; lr=1.623e-04; interval_elapsed=219.5s; total_time=582.6m
[step 38800] avg_loss=5.2106; lr=1.617e-04; interval_elapsed=218.5s; total_time=586.3m
[step 39000] avg_loss=5.4233; lr=1.612e-04; interval_elapsed=218.2s; total_time=589.9m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-39000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 39000] val_loss=5.4729
[step 39200] avg_loss=5.3140; lr=1.607e-04; interval_elapsed=227.6s; total_time=593.7m
[step 39400] avg_loss=5.2171; lr=1.602e-04; interval_elapsed=218.3s; total_time=597.3m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-39500.pt
[step 39600] avg_loss=5.2861; lr=1.596e-04; interval_elapsed=218.1s; total_time=601.0m
[step 39800] avg_loss=5.4430; lr=1.591e-04; interval_elapsed=218.3s; total_time=604.6m
[step 40000] avg_loss=5.2083; lr=1.586e-04; interval_elapsed=218.0s; total_time=608.3m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-40000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 40000] val_loss=5.5435
[step 40200] avg_loss=6.2754; lr=1.580e-04; interval_elapsed=227.8s; total_time=612.1m
[step 40400] avg_loss=6.3611; lr=1.575e-04; interval_elapsed=217.9s; total_time=615.7m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-40500.pt
[step 40600] avg_loss=6.3572; lr=1.569e-04; interval_elapsed=218.1s; total_time=619.3m
[step 40800] avg_loss=5.6408; lr=1.564e-04; interval_elapsed=218.0s; total_time=623.0m
[step 41000] avg_loss=5.3266; lr=1.558e-04; interval_elapsed=218.1s; total_time=626.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-41000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 41000] val_loss=5.4768
[step 41200] avg_loss=5.4712; lr=1.553e-04; interval_elapsed=228.3s; total_time=630.4m
[step 41400] avg_loss=5.4773; lr=1.547e-04; interval_elapsed=218.3s; total_time=634.0m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-41500.pt
[step 41600] avg_loss=5.4708; lr=1.542e-04; interval_elapsed=218.6s; total_time=637.7m
[step 41800] avg_loss=5.4605; lr=1.536e-04; interval_elapsed=218.5s; total_time=641.3m
[step 42000] avg_loss=5.4630; lr=1.530e-04; interval_elapsed=218.4s; total_time=645.0m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-42000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 42000] val_loss=5.4650
[step 42200] avg_loss=5.4223; lr=1.525e-04; interval_elapsed=228.2s; total_time=648.8m
[step 42400] avg_loss=5.2256; lr=1.519e-04; interval_elapsed=218.5s; total_time=652.4m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-42500.pt
[step 42600] avg_loss=5.4663; lr=1.514e-04; interval_elapsed=218.7s; total_time=656.0m
[step 42800] avg_loss=5.4563; lr=1.508e-04; interval_elapsed=218.7s; total_time=659.7m
[step 43000] avg_loss=5.4680; lr=1.502e-04; interval_elapsed=218.6s; total_time=663.3m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-43000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 43000] val_loss=5.4604
[step 43200] avg_loss=5.4112; lr=1.496e-04; interval_elapsed=228.3s; total_time=667.1m
[step 43400] avg_loss=5.2317; lr=1.491e-04; interval_elapsed=218.6s; total_time=670.8m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-43500.pt
[step 43600] avg_loss=5.4649; lr=1.485e-04; interval_elapsed=218.7s; total_time=674.4m
[step 43800] avg_loss=5.4521; lr=1.479e-04; interval_elapsed=218.5s; total_time=678.1m
[step 44000] avg_loss=5.4348; lr=1.473e-04; interval_elapsed=218.6s; total_time=681.7m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-44000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 44000] val_loss=5.4567
[step 44200] avg_loss=5.4606; lr=1.467e-04; interval_elapsed=228.4s; total_time=685.5m
[step 44400] avg_loss=5.4623; lr=1.461e-04; interval_elapsed=219.1s; total_time=689.2m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-44500.pt
[step 44600] avg_loss=5.4547; lr=1.456e-04; interval_elapsed=218.7s; total_time=692.8m
[step 44800] avg_loss=5.2868; lr=1.450e-04; interval_elapsed=218.5s; total_time=696.5m
[step 45000] avg_loss=5.3425; lr=1.444e-04; interval_elapsed=218.9s; total_time=700.1m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-45000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 45000] val_loss=5.4542
[step 45200] avg_loss=5.7094; lr=1.438e-04; interval_elapsed=228.8s; total_time=703.9m
[step 45400] avg_loss=6.3528; lr=1.432e-04; interval_elapsed=218.5s; total_time=707.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-45500.pt
[step 45600] avg_loss=6.3193; lr=1.426e-04; interval_elapsed=219.2s; total_time=711.2m
[step 45800] avg_loss=6.3232; lr=1.420e-04; interval_elapsed=218.5s; total_time=714.9m
[step 46000] avg_loss=5.2606; lr=1.414e-04; interval_elapsed=218.5s; total_time=718.5m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-46000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 46000] val_loss=5.5237

[step 46200] avg_loss=5.2116; lr=1.408e-04; interval_elapsed=239.5s; total_time=722.5m
[step 46400] avg_loss=5.2000; lr=1.402e-04; interval_elapsed=270.8s; total_time=727.0m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-46500.pt
[step 46600] avg_loss=5.3266; lr=1.396e-04; interval_elapsed=245.4s; total_time=731.1m
[step 46800] avg_loss=5.4481; lr=1.389e-04; interval_elapsed=223.7s; total_time=734.8m
[step 47000] avg_loss=5.4525; lr=1.383e-04; interval_elapsed=230.3s; total_time=738.7m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-47000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 47000] val_loss=5.4548
[step 47200] avg_loss=5.4442; lr=1.377e-04; interval_elapsed=241.6s; total_time=742.7m
[step 47400] avg_loss=6.0267; lr=1.371e-04; interval_elapsed=229.6s; total_time=746.5m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-47500.pt
[step 47600] avg_loss=6.3172; lr=1.365e-04; interval_elapsed=236.5s; total_time=750.5m
[step 47800] avg_loss=6.2907; lr=1.359e-04; interval_elapsed=247.2s; total_time=754.6m
[step 48000] avg_loss=5.9324; lr=1.352e-04; interval_elapsed=296.8s; total_time=759.5m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-48000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 48000] val_loss=5.5218
[step 48200] avg_loss=5.2301; lr=1.346e-04; interval_elapsed=258.3s; total_time=763.8m
[step 48400] avg_loss=5.2154; lr=1.340e-04; interval_elapsed=238.3s; total_time=767.8m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-48500.pt
[step 48600] avg_loss=5.4447; lr=1.334e-04; interval_elapsed=243.4s; total_time=771.9m
[step 48800] avg_loss=5.4403; lr=1.327e-04; interval_elapsed=250.1s; total_time=776.0m
[step 49000] avg_loss=5.4550; lr=1.321e-04; interval_elapsed=233.0s; total_time=779.9m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-49000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 49000] val_loss=5.4527
[step 49200] avg_loss=5.3815; lr=1.315e-04; interval_elapsed=240.5s; total_time=783.9m
[step 49400] avg_loss=5.2355; lr=1.309e-04; interval_elapsed=228.3s; total_time=787.7m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-49500.pt
[step 49600] avg_loss=5.4560; lr=1.302e-04; interval_elapsed=226.5s; total_time=791.5m
[step 49800] avg_loss=5.2386; lr=1.296e-04; interval_elapsed=225.2s; total_time=795.3m
[step 50000] avg_loss=5.3725; lr=1.290e-04; interval_elapsed=226.9s; total_time=799.0m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-50000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 50000] val_loss=5.4524
[step 50200] avg_loss=5.4367; lr=1.283e-04; interval_elapsed=235.2s; total_time=803.0m
[step 50400] avg_loss=5.4156; lr=1.277e-04; interval_elapsed=225.5s; total_time=806.7m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-50500.pt
[step 50600] avg_loss=5.4470; lr=1.271e-04; interval_elapsed=225.6s; total_time=810.5m
[step 50800] avg_loss=5.4359; lr=1.264e-04; interval_elapsed=227.0s; total_time=814.3m
[step 51000] avg_loss=5.3048; lr=1.258e-04; interval_elapsed=227.3s; total_time=818.0m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-51000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 51000] val_loss=5.4953
[step 51200] avg_loss=5.2958; lr=1.251e-04; interval_elapsed=235.7s; total_time=822.0m
[step 51400] avg_loss=5.4270; lr=1.245e-04; interval_elapsed=226.8s; total_time=825.8m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-51500.pt
[step 51600] avg_loss=5.4464; lr=1.238e-04; interval_elapsed=228.9s; total_time=829.6m
[step 51800] avg_loss=5.2294; lr=1.232e-04; interval_elapsed=227.0s; total_time=833.4m
[step 52000] avg_loss=5.1704; lr=1.226e-04; interval_elapsed=228.3s; total_time=837.2m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-52000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 52000] val_loss=5.4981
[step 52200] avg_loss=5.8540; lr=1.219e-04; interval_elapsed=236.2s; total_time=841.1m
[step 52400] avg_loss=6.3074; lr=1.213e-04; interval_elapsed=227.1s; total_time=844.9m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-52500.pt
[step 52600] avg_loss=6.2882; lr=1.206e-04; interval_elapsed=246.3s; total_time=849.0m
[step 52800] avg_loss=6.0331; lr=1.200e-04; interval_elapsed=311.9s; total_time=854.2m
[step 53000] avg_loss=5.1982; lr=1.193e-04; interval_elapsed=242.4s; total_time=858.2m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-53000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 53000] val_loss=5.4631
[step 53200] avg_loss=5.4279; lr=1.187e-04; interval_elapsed=327.9s; total_time=863.7m
[step 53400] avg_loss=5.4415; lr=1.180e-04; interval_elapsed=295.6s; total_time=868.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-53500.pt
[step 53600] avg_loss=5.3085; lr=1.174e-04; interval_elapsed=231.7s; total_time=872.5m
[step 53800] avg_loss=5.2906; lr=1.167e-04; interval_elapsed=232.6s; total_time=876.4m
[step 54000] avg_loss=5.4291; lr=1.160e-04; interval_elapsed=244.7s; total_time=880.4m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-54000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 54000] val_loss=5.4757
[step 54200] avg_loss=5.1657; lr=1.154e-04; interval_elapsed=266.4s; total_time=884.9m
[step 54400] avg_loss=6.1933; lr=1.147e-04; interval_elapsed=250.4s; total_time=889.0m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-54500.pt
[step 54600] avg_loss=6.2858; lr=1.141e-04; interval_elapsed=261.5s; total_time=893.4m
[step 54800] avg_loss=6.2775; lr=1.134e-04; interval_elapsed=285.5s; total_time=898.2m
[step 55000] avg_loss=5.7567; lr=1.128e-04; interval_elapsed=291.8s; total_time=903.0m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-55000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 55000] val_loss=5.4531
[step 55200] avg_loss=5.4497; lr=1.121e-04; interval_elapsed=307.3s; total_time=908.1m
[step 55400] avg_loss=5.4494; lr=1.115e-04; interval_elapsed=335.3s; total_time=913.7m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-55500.pt
[step 55600] avg_loss=5.2116; lr=1.108e-04; interval_elapsed=412.9s; total_time=920.6m
[step 55800] avg_loss=5.4261; lr=1.101e-04; interval_elapsed=310.8s; total_time=925.8m
[step 56000] avg_loss=5.4489; lr=1.095e-04; interval_elapsed=256.1s; total_time=930.1m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-56000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 56000] val_loss=5.4482
[step 56200] avg_loss=5.3478; lr=1.088e-04; interval_elapsed=279.1s; total_time=934.7m
[step 56400] avg_loss=5.2287; lr=1.082e-04; interval_elapsed=304.9s; total_time=939.8m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-56500.pt
[step 56600] avg_loss=5.4467; lr=1.075e-04; interval_elapsed=279.7s; total_time=944.5m
[step 56800] avg_loss=6.2257; lr=1.068e-04; interval_elapsed=290.0s; total_time=949.3m
[step 57000] avg_loss=6.2673; lr=1.062e-04; interval_elapsed=258.8s; total_time=953.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-57000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 57000] val_loss=5.5468
[step 57200] avg_loss=6.2623; lr=1.055e-04; interval_elapsed=256.2s; total_time=957.9m
[step 57400] avg_loss=5.7687; lr=1.048e-04; interval_elapsed=250.7s; total_time=962.1m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-57500.pt
[step 57600] avg_loss=5.3566; lr=1.042e-04; interval_elapsed=244.5s; total_time=966.1m
[step 57800] avg_loss=5.2527; lr=1.035e-04; interval_elapsed=264.5s; total_time=970.5m
[step 58000] avg_loss=5.4501; lr=1.029e-04; interval_elapsed=244.0s; total_time=974.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-58000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 58000] val_loss=5.4486
[step 58200] avg_loss=5.4076; lr=1.022e-04; interval_elapsed=267.5s; total_time=979.1m
[step 58400] avg_loss=5.2675; lr=1.015e-04; interval_elapsed=253.8s; total_time=983.3m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-58500.pt
[step 58600] avg_loss=5.3362; lr=1.009e-04; interval_elapsed=242.5s; total_time=987.3m
[step 58800] avg_loss=5.4367; lr=1.002e-04; interval_elapsed=239.8s; total_time=991.3m
[step 59000] avg_loss=5.4280; lr=9.954e-05; interval_elapsed=258.4s; total_time=995.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-59000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 59000] val_loss=5.4525
[step 59200] avg_loss=5.1842; lr=9.887e-05; interval_elapsed=251.3s; total_time=999.8m
[step 59400] avg_loss=5.1725; lr=9.821e-05; interval_elapsed=248.4s; total_time=1004.0m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-59500.pt
[step 59600] avg_loss=5.3779; lr=9.755e-05; interval_elapsed=249.7s; total_time=1008.1m
[step 59800] avg_loss=5.3178; lr=9.688e-05; interval_elapsed=259.4s; total_time=1012.5m
[step 60000] avg_loss=5.1811; lr=9.622e-05; interval_elapsed=244.5s; total_time=1016.5m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-60000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 60000] val_loss=5.4734
[step 60200] avg_loss=5.2034; lr=9.556e-05; interval_elapsed=260.0s; total_time=1020.9m
[step 60400] avg_loss=5.4322; lr=9.490e-05; interval_elapsed=240.5s; total_time=1024.9m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-60500.pt
[step 60600] avg_loss=5.2126; lr=9.423e-05; interval_elapsed=243.2s; total_time=1028.9m
[step 60800] avg_loss=5.1890; lr=9.357e-05; interval_elapsed=245.2s; total_time=1033.0m
[step 61000] avg_loss=5.3227; lr=9.291e-05; interval_elapsed=243.7s; total_time=1037.1m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-61000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 61000] val_loss=5.4487

[step 61200] avg_loss=5.6473; lr=9.225e-05; interval_elapsed=289.5s; total_time=1041.9m
[step 61400] avg_loss=6.2829; lr=9.159e-05; interval_elapsed=254.1s; total_time=1046.1m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-61500.pt
[step 61600] avg_loss=6.2672; lr=9.093e-05; interval_elapsed=250.8s; total_time=1050.3m
[step 61800] avg_loss=6.2253; lr=9.027e-05; interval_elapsed=241.4s; total_time=1054.3m
[step 62000] avg_loss=5.1790; lr=8.961e-05; interval_elapsed=243.4s; total_time=1058.4m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-62000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 62000] val_loss=5.4797
[step 62200] avg_loss=5.3976; lr=8.895e-05; interval_elapsed=251.9s; total_time=1062.6m
[step 62400] avg_loss=5.4465; lr=8.829e-05; interval_elapsed=241.4s; total_time=1066.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-62500.pt
[step 62600] avg_loss=5.3514; lr=8.763e-05; interval_elapsed=241.5s; total_time=1070.6m
[step 62800] avg_loss=5.3432; lr=8.697e-05; interval_elapsed=240.7s; total_time=1074.6m
[step 63000] avg_loss=6.2643; lr=8.631e-05; interval_elapsed=238.9s; total_time=1078.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-63000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 63000] val_loss=5.5038
[step 63200] avg_loss=6.2649; lr=8.566e-05; interval_elapsed=252.5s; total_time=1082.8m
[step 63400] avg_loss=6.2690; lr=8.500e-05; interval_elapsed=237.4s; total_time=1086.8m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-63500.pt
[step 63600] avg_loss=5.5263; lr=8.435e-05; interval_elapsed=239.1s; total_time=1090.8m
[step 63800] avg_loss=5.4407; lr=8.369e-05; interval_elapsed=237.7s; total_time=1094.7m
[step 64000] avg_loss=5.3366; lr=8.304e-05; interval_elapsed=241.4s; total_time=1098.8m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-64000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 64000] val_loss=5.4586
[step 64200] avg_loss=5.2355; lr=8.238e-05; interval_elapsed=253.6s; total_time=1103.0m
[step 64400] avg_loss=5.4401; lr=8.173e-05; interval_elapsed=243.9s; total_time=1107.1m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-64500.pt
[step 64600] avg_loss=5.4373; lr=8.108e-05; interval_elapsed=244.1s; total_time=1111.1m
[step 64800] avg_loss=5.2831; lr=8.043e-05; interval_elapsed=248.0s; total_time=1115.3m
[step 65000] avg_loss=5.3200; lr=7.978e-05; interval_elapsed=303.4s; total_time=1120.3m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-65000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 65000] val_loss=5.4467
[step 65200] avg_loss=5.4475; lr=7.913e-05; interval_elapsed=256.4s; total_time=1124.6m
[step 65400] avg_loss=5.4462; lr=7.848e-05; interval_elapsed=245.3s; total_time=1128.7m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-65500.pt
[step 65600] avg_loss=5.4404; lr=7.783e-05; interval_elapsed=251.8s; total_time=1132.9m
[step 65800] avg_loss=5.4193; lr=7.719e-05; interval_elapsed=252.8s; total_time=1137.1m
[step 66000] avg_loss=5.2991; lr=7.654e-05; interval_elapsed=239.2s; total_time=1141.1m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-66000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 66000] val_loss=5.4570
[step 66200] avg_loss=5.2937; lr=7.590e-05; interval_elapsed=250.5s; total_time=1145.2m
[step 66400] avg_loss=5.5304; lr=7.526e-05; interval_elapsed=240.3s; total_time=1149.3m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-66500.pt
[step 66600] avg_loss=6.2535; lr=7.461e-05; interval_elapsed=238.6s; total_time=1153.2m
[step 66800] avg_loss=6.2500; lr=7.397e-05; interval_elapsed=238.9s; total_time=1157.2m
[step 67000] avg_loss=6.2393; lr=7.333e-05; interval_elapsed=239.7s; total_time=1161.2m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-67000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 67000] val_loss=5.5014
[step 67200] avg_loss=5.6364; lr=7.269e-05; interval_elapsed=249.0s; total_time=1165.4m
[step 67400] avg_loss=5.3434; lr=7.206e-05; interval_elapsed=239.3s; total_time=1169.3m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-67500.pt
[step 67600] avg_loss=5.2939; lr=7.142e-05; interval_elapsed=238.2s; total_time=1173.3m
[step 67800] avg_loss=5.5175; lr=7.078e-05; interval_elapsed=238.2s; total_time=1177.3m
[step 68000] avg_loss=6.2699; lr=7.015e-05; interval_elapsed=238.3s; total_time=1181.3m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-68000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 68000] val_loss=5.4705
[step 68200] avg_loss=6.2631; lr=6.952e-05; interval_elapsed=249.1s; total_time=1185.4m
[step 68400] avg_loss=6.2596; lr=6.889e-05; interval_elapsed=240.6s; total_time=1189.4m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-68500.pt
[step 68600] avg_loss=6.2581; lr=6.826e-05; interval_elapsed=238.7s; total_time=1193.4m
[step 68800] avg_loss=6.2283; lr=6.763e-05; interval_elapsed=240.1s; total_time=1197.4m
[step 69000] avg_loss=6.2670; lr=6.700e-05; interval_elapsed=238.9s; total_time=1201.4m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-69000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 69000] val_loss=5.5030
[step 69200] avg_loss=5.8788; lr=6.638e-05; interval_elapsed=248.5s; total_time=1205.5m
[step 69400] avg_loss=5.3840; lr=6.575e-05; interval_elapsed=238.4s; total_time=1209.5m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-69500.pt
[step 69600] avg_loss=5.1830; lr=6.513e-05; interval_elapsed=239.5s; total_time=1213.5m
[step 69800] avg_loss=5.4393; lr=6.451e-05; interval_elapsed=239.8s; total_time=1217.5m
[step 70000] avg_loss=5.2570; lr=6.389e-05; interval_elapsed=237.9s; total_time=1221.4m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-70000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 70000] val_loss=5.4530
[step 70200] avg_loss=5.1823; lr=6.327e-05; interval_elapsed=247.8s; total_time=1225.6m
[step 70400] avg_loss=5.1969; lr=6.266e-05; interval_elapsed=240.4s; total_time=1229.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-70500.pt
[step 70600] avg_loss=5.2648; lr=6.204e-05; interval_elapsed=247.7s; total_time=1233.7m
[step 70800] avg_loss=5.4414; lr=6.143e-05; interval_elapsed=239.3s; total_time=1237.7m
[step 71000] avg_loss=5.4452; lr=6.082e-05; interval_elapsed=237.8s; total_time=1241.7m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-71000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 71000] val_loss=5.4463
[step 71200] avg_loss=5.2642; lr=6.021e-05; interval_elapsed=247.4s; total_time=1245.8m
[step 71400] avg_loss=5.3290; lr=5.960e-05; interval_elapsed=236.5s; total_time=1249.7m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-71500.pt
[step 71600] avg_loss=5.4390; lr=5.900e-05; interval_elapsed=235.7s; total_time=1253.7m
[step 71800] avg_loss=5.4262; lr=5.839e-05; interval_elapsed=235.9s; total_time=1257.6m
[step 72000] avg_loss=5.1596; lr=5.779e-05; interval_elapsed=235.5s; total_time=1261.5m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-72000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 72000] val_loss=5.4522
[step 72200] avg_loss=5.1728; lr=5.719e-05; interval_elapsed=249.6s; total_time=1265.7m
[step 72400] avg_loss=5.8793; lr=5.659e-05; interval_elapsed=237.1s; total_time=1269.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-72500.pt
[step 72600] avg_loss=6.2446; lr=5.599e-05; interval_elapsed=235.3s; total_time=1273.5m
[step 72800] avg_loss=6.2499; lr=5.540e-05; interval_elapsed=244.3s; total_time=1277.6m
[step 73000] avg_loss=5.8346; lr=5.481e-05; interval_elapsed=239.4s; total_time=1281.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-73000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 73000] val_loss=5.4595
[step 73200] avg_loss=5.4225; lr=5.422e-05; interval_elapsed=247.3s; total_time=1285.7m
[step 73400] avg_loss=6.2633; lr=5.363e-05; interval_elapsed=241.0s; total_time=1289.7m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-73500.pt
[step 73600] avg_loss=6.2437; lr=5.304e-05; interval_elapsed=238.3s; total_time=1293.7m
[step 73800] avg_loss=6.2417; lr=5.246e-05; interval_elapsed=235.6s; total_time=1297.6m
[step 74000] avg_loss=6.2713; lr=5.187e-05; interval_elapsed=235.1s; total_time=1301.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-74000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 74000] val_loss=5.4850
[step 74200] avg_loss=6.2413; lr=5.129e-05; interval_elapsed=245.3s; total_time=1305.7m
[step 74400] avg_loss=6.2397; lr=5.071e-05; interval_elapsed=234.4s; total_time=1309.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-74500.pt
[step 74600] avg_loss=5.8094; lr=5.014e-05; interval_elapsed=233.1s; total_time=1313.4m
[step 74800] avg_loss=5.6456; lr=4.957e-05; interval_elapsed=234.2s; total_time=1317.3m
[step 75000] avg_loss=6.2515; lr=4.899e-05; interval_elapsed=233.4s; total_time=1321.2m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-75000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 75000] val_loss=5.4595
[step 75200] avg_loss=6.2443; lr=4.842e-05; interval_elapsed=243.1s; total_time=1325.3m
[step 75400] avg_loss=6.2380; lr=4.786e-05; interval_elapsed=232.7s; total_time=1329.2m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-75500.pt
[step 75600] avg_loss=5.2491; lr=4.729e-05; interval_elapsed=235.1s; total_time=1333.1m
[step 75800] avg_loss=5.1788; lr=4.673e-05; interval_elapsed=239.6s; total_time=1337.1m
[step 76000] avg_loss=5.1770; lr=4.617e-05; interval_elapsed=251.4s; total_time=1341.3m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-76000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 76000] val_loss=5.4593
[step 76200] avg_loss=5.2036; lr=4.561e-05; interval_elapsed=255.8s; total_time=1345.5m
[step 76400] avg_loss=5.2601; lr=4.506e-05; interval_elapsed=249.8s; total_time=1349.7m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-76500.pt
[step 76600] avg_loss=5.4444; lr=4.450e-05; interval_elapsed=245.6s; total_time=1353.8m
[step 76800] avg_loss=5.4252; lr=4.395e-05; interval_elapsed=253.1s; total_time=1358.0m
[step 77000] avg_loss=5.2557; lr=4.340e-05; interval_elapsed=245.9s; total_time=1362.1m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-77000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 77000] val_loss=5.4515
[step 77200] avg_loss=5.3393; lr=4.286e-05; interval_elapsed=291.5s; total_time=1367.0m
[step 77400] avg_loss=5.4444; lr=4.232e-05; interval_elapsed=263.5s; total_time=1371.4m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-77500.pt
[step 77600] avg_loss=5.4399; lr=4.178e-05; interval_elapsed=263.0s; total_time=1375.7m
[step 77800] avg_loss=5.4374; lr=4.124e-05; interval_elapsed=263.4s; total_time=1380.1m
[step 78000] avg_loss=5.4406; lr=4.070e-05; interval_elapsed=263.3s; total_time=1384.5m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-78000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 78000] val_loss=5.4474
[step 78200] avg_loss=5.4346; lr=4.017e-05; interval_elapsed=271.7s; total_time=1389.0m
[step 78400] avg_loss=5.3910; lr=3.964e-05; interval_elapsed=263.3s; total_time=1393.4m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-78500.pt
[step 78600] avg_loss=5.2056; lr=3.911e-05; interval_elapsed=269.3s; total_time=1397.9m
[step 78800] avg_loss=6.2704; lr=3.859e-05; interval_elapsed=260.3s; total_time=1402.3m
[step 79000] avg_loss=6.2632; lr=3.806e-05; interval_elapsed=261.3s; total_time=1406.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-79000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 79000] val_loss=5.4624
[step 79200] avg_loss=6.2328; lr=3.755e-05; interval_elapsed=274.5s; total_time=1411.2m
[step 79400] avg_loss=5.5374; lr=3.703e-05; interval_elapsed=245.6s; total_time=1415.3m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-79500.pt
[step 79600] avg_loss=5.1802; lr=3.651e-05; interval_elapsed=281.9s; total_time=1420.0m
[step 79800] avg_loss=5.2685; lr=3.600e-05; interval_elapsed=251.5s; total_time=1424.2m
[step 80000] avg_loss=5.4482; lr=3.550e-05; interval_elapsed=262.2s; total_time=1428.5m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-80000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 80000] val_loss=5.4484
[step 80200] avg_loss=5.4447; lr=3.499e-05; interval_elapsed=276.7s; total_time=1433.2m
[step 80400] avg_loss=5.2478; lr=3.449e-05; interval_elapsed=275.9s; total_time=1437.8m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-80500.pt
[step 80600] avg_loss=5.1679; lr=3.399e-05; interval_elapsed=260.7s; total_time=1442.1m
[step 80800] avg_loss=5.2851; lr=3.349e-05; interval_elapsed=255.5s; total_time=1446.4m
[step 81000] avg_loss=5.4361; lr=3.300e-05; interval_elapsed=228.3s; total_time=1450.2m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-81000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 81000] val_loss=5.4479
[step 81200] avg_loss=5.4419; lr=3.251e-05; interval_elapsed=250.9s; total_time=1454.3m
[step 81400] avg_loss=5.1918; lr=3.202e-05; interval_elapsed=250.7s; total_time=1458.5m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-81500.pt
[step 81600] avg_loss=5.3758; lr=3.153e-05; interval_elapsed=248.1s; total_time=1462.7m
[step 81800] avg_loss=5.4467; lr=3.105e-05; interval_elapsed=256.5s; total_time=1466.9m
[step 82000] avg_loss=5.3765; lr=3.057e-05; interval_elapsed=254.3s; total_time=1471.2m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-82000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 82000] val_loss=5.4478
[step 82200] avg_loss=5.1855; lr=3.010e-05; interval_elapsed=259.0s; total_time=1475.5m
[step 82400] avg_loss=5.1763; lr=2.962e-05; interval_elapsed=248.9s; total_time=1479.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-82500.pt
[step 82600] avg_loss=5.4191; lr=2.915e-05; interval_elapsed=249.3s; total_time=1483.8m
[step 82800] avg_loss=5.4390; lr=2.869e-05; interval_elapsed=250.1s; total_time=1488.0m
[step 83000] avg_loss=5.4401; lr=2.822e-05; interval_elapsed=249.0s; total_time=1492.1m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-83000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 83000] val_loss=5.4470
[step 83200] avg_loss=5.4520; lr=2.776e-05; interval_elapsed=259.8s; total_time=1496.4m
[step 83400] avg_loss=5.4444; lr=2.731e-05; interval_elapsed=249.5s; total_time=1500.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-83500.pt
[step 83600] avg_loss=5.4497; lr=2.685e-05; interval_elapsed=248.7s; total_time=1504.7m
[step 83800] avg_loss=5.2652; lr=2.640e-05; interval_elapsed=267.7s; total_time=1509.2m
[step 84000] avg_loss=5.1791; lr=2.595e-05; interval_elapsed=247.8s; total_time=1513.3m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-84000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 84000] val_loss=5.4482
[step 84200] avg_loss=5.2626; lr=2.551e-05; interval_elapsed=265.2s; total_time=1517.8m
[step 84400] avg_loss=5.4411; lr=2.507e-05; interval_elapsed=304.1s; total_time=1522.8m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-84500.pt
[step 84600] avg_loss=5.4397; lr=2.463e-05; interval_elapsed=409.6s; total_time=1529.7m
[step 84800] avg_loss=6.0624; lr=2.420e-05; interval_elapsed=360.6s; total_time=1535.7m
[step 85000] avg_loss=6.2612; lr=2.377e-05; interval_elapsed=372.1s; total_time=1541.9m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-85000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 85000] val_loss=5.4494
[step 85200] avg_loss=6.2481; lr=2.334e-05; interval_elapsed=337.1s; total_time=1547.5m
[step 85400] avg_loss=5.7827; lr=2.291e-05; interval_elapsed=269.8s; total_time=1552.0m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-85500.pt
[step 85600] avg_loss=5.2747; lr=2.249e-05; interval_elapsed=256.6s; total_time=1556.3m
[step 85800] avg_loss=5.4538; lr=2.208e-05; interval_elapsed=251.4s; total_time=1560.4m
[step 86000] avg_loss=5.4460; lr=2.166e-05; interval_elapsed=241.0s; total_time=1564.5m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-86000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 86000] val_loss=5.4474
[step 86200] avg_loss=5.2550; lr=2.125e-05; interval_elapsed=260.9s; total_time=1568.8m
[step 86400] avg_loss=5.3543; lr=2.085e-05; interval_elapsed=253.4s; total_time=1573.0m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-86500.pt
[step 86600] avg_loss=5.4527; lr=2.044e-05; interval_elapsed=251.7s; total_time=1577.2m
[step 86800] avg_loss=5.4483; lr=2.004e-05; interval_elapsed=267.7s; total_time=1581.7m
[step 87000] avg_loss=5.2218; lr=1.964e-05; interval_elapsed=259.7s; total_time=1586.0m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-87000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 87000] val_loss=5.4472
[step 87200] avg_loss=5.1783; lr=1.925e-05; interval_elapsed=281.2s; total_time=1590.7m
[step 87400] avg_loss=5.9830; lr=1.886e-05; interval_elapsed=260.3s; total_time=1595.0m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-87500.pt
[step 87600] avg_loss=6.2464; lr=1.848e-05; interval_elapsed=261.7s; total_time=1599.4m
[step 87800] avg_loss=6.2445; lr=1.809e-05; interval_elapsed=277.6s; total_time=1604.0m
[step 88000] avg_loss=5.8015; lr=1.772e-05; interval_elapsed=271.4s; total_time=1608.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-88000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 88000] val_loss=5.4490
[step 88200] avg_loss=5.2478; lr=1.734e-05; interval_elapsed=285.2s; total_time=1613.3m
[step 88400] avg_loss=5.4445; lr=1.697e-05; interval_elapsed=286.0s; total_time=1618.1m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-88500.pt
[step 88600] avg_loss=5.1758; lr=1.660e-05; interval_elapsed=276.4s; total_time=1622.7m
[step 88800] avg_loss=5.2029; lr=1.624e-05; interval_elapsed=280.4s; total_time=1627.4m
[step 89000] avg_loss=5.3677; lr=1.588e-05; interval_elapsed=298.6s; total_time=1632.3m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-89000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 89000] val_loss=5.4475
[step 89200] avg_loss=5.3284; lr=1.552e-05; interval_elapsed=295.4s; total_time=1637.3m
[step 89400] avg_loss=5.2444; lr=1.517e-05; interval_elapsed=294.8s; total_time=1642.2m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-89500.pt
[step 89600] avg_loss=5.4447; lr=1.482e-05; interval_elapsed=274.2s; total_time=1646.7m
[step 89800] avg_loss=5.4436; lr=1.447e-05; interval_elapsed=270.1s; total_time=1651.2m
[step 90000] avg_loss=5.4430; lr=1.413e-05; interval_elapsed=293.4s; total_time=1656.1m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-90000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 90000] val_loss=5.4473
[step 90200] avg_loss=5.3199; lr=1.379e-05; interval_elapsed=269.3s; total_time=1660.6m
[step 90400] avg_loss=5.1906; lr=1.346e-05; interval_elapsed=269.9s; total_time=1665.1m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-90500.pt
[step 90600] avg_loss=5.2607; lr=1.313e-05; interval_elapsed=254.6s; total_time=1669.4m
[step 90800] avg_loss=5.4656; lr=1.280e-05; interval_elapsed=238.3s; total_time=1673.3m
[step 91000] avg_loss=5.2086; lr=1.248e-05; interval_elapsed=247.0s; total_time=1677.5m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-91000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 91000] val_loss=5.4474
[step 91200] avg_loss=5.3780; lr=1.216e-05; interval_elapsed=257.7s; total_time=1681.7m
[step 91400] avg_loss=5.3283; lr=1.184e-05; interval_elapsed=275.5s; total_time=1686.3m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-91500.pt
[step 91600] avg_loss=5.5190; lr=1.153e-05; interval_elapsed=255.0s; total_time=1690.6m
[step 91800] avg_loss=6.2438; lr=1.123e-05; interval_elapsed=229.7s; total_time=1694.4m
[step 92000] avg_loss=6.2332; lr=1.092e-05; interval_elapsed=233.7s; total_time=1698.3m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-92000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 92000] val_loss=5.4478
[step 92200] avg_loss=6.2262; lr=1.062e-05; interval_elapsed=240.2s; total_time=1702.3m
[step 92400] avg_loss=5.1951; lr=1.033e-05; interval_elapsed=223.1s; total_time=1706.0m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-92500.pt
[step 92600] avg_loss=5.2170; lr=1.004e-05; interval_elapsed=223.9s; total_time=1709.8m
[step 92800] avg_loss=5.3569; lr=9.748e-06; interval_elapsed=222.8s; total_time=1713.5m
[step 93000] avg_loss=5.3497; lr=9.464e-06; interval_elapsed=223.0s; total_time=1717.2m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-93000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 93000] val_loss=5.4475
[step 93200] avg_loss=5.1980; lr=9.184e-06; interval_elapsed=231.8s; total_time=1721.1m
[step 93400] avg_loss=5.1615; lr=8.909e-06; interval_elapsed=222.6s; total_time=1724.8m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-93500.pt
[step 93600] avg_loss=5.1615; lr=8.637e-06; interval_elapsed=224.9s; total_time=1728.5m
[step 93800] avg_loss=5.1979; lr=8.369e-06; interval_elapsed=226.8s; total_time=1732.3m
[step 94000] avg_loss=5.1938; lr=8.106e-06; interval_elapsed=224.0s; total_time=1736.0m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-94000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a790>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 94000] val_loss=5.4480
[step 94200] avg_loss=5.3584; lr=7.846e-06; interval_elapsed=228.9s; total_time=1739.8m
[step 94400] avg_loss=5.4361; lr=7.591e-06; interval_elapsed=219.6s; total_time=1743.5m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-94500.pt
[step 94600] avg_loss=5.3909; lr=7.339e-06; interval_elapsed=228.5s; total_time=1747.3m
[step 94800] avg_loss=5.1694; lr=7.092e-06; interval_elapsed=227.1s; total_time=1751.1m
[step 95000] avg_loss=5.1852; lr=6.849e-06; interval_elapsed=225.6s; total_time=1754.9m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-95000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 95000] val_loss=5.4476
[step 95200] avg_loss=5.2098; lr=6.609e-06; interval_elapsed=236.4s; total_time=1758.8m
[step 95400] avg_loss=5.1837; lr=6.374e-06; interval_elapsed=225.4s; total_time=1762.6m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-95500.pt
[step 95600] avg_loss=5.1739; lr=6.143e-06; interval_elapsed=225.2s; total_time=1766.3m
[step 95800] avg_loss=5.1954; lr=5.916e-06; interval_elapsed=227.2s; total_time=1770.1m
[step 96000] avg_loss=5.2051; lr=5.694e-06; interval_elapsed=225.6s; total_time=1773.9m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-96000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a350>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 96000] val_loss=5.4478
[step 96200] avg_loss=5.1983; lr=5.475e-06; interval_elapsed=235.5s; total_time=1777.8m
[step 96400] avg_loss=5.1866; lr=5.261e-06; interval_elapsed=225.7s; total_time=1781.5m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-96500.pt
[step 96600] avg_loss=5.3981; lr=5.051e-06; interval_elapsed=226.0s; total_time=1785.3m
[step 96800] avg_loss=5.4312; lr=4.845e-06; interval_elapsed=226.4s; total_time=1789.1m
[step 97000] avg_loss=5.6748; lr=4.643e-06; interval_elapsed=226.0s; total_time=1792.8m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-97000.pt
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a570>
Traceback (most recent call last):
  File "<python-input-0>", line 215, in <module>
RuntimeError: generator ignored GeneratorExit
[validation @ step 97000] val_loss=5.4477
[step 97200] avg_loss=6.2497; lr=4.445e-06; interval_elapsed=235.3s; total_time=1796.8m
[step 97400] avg_loss=6.2497; lr=4.252e-06; interval_elapsed=226.3s; total_time=1800.5m
Saved checkpoint: ./data/gemma20m_ckpts/ckpt-97500.pt
[step 97600] avg_loss=6.2448; lr=4.062e-06; interval_elapsed=226.3s; total_time=1804.3m
Exception ignored in: <generator object TokLineChunker._line_generator at 0x12ad7a240>
Traceback (most recent call last):
  File "<python-input-0>", line 223, in <module>
RuntimeError: generator ignored GeneratorExit
Training finished. Final model saved to ./data/gemma20m_ckpts/final.pt
```

#### Testing Results

Best checkpoint: ckpt-83000.pt

Because the validation loss is lowest at this point and it generalizes better

While at 95k checkpoint there is repeatition and memorization

In [ ]:
import torch
from transformers import AutoConfig, AutoModelForCausalLM
import sentencepiece as spm

# 1. Tokenizer
sp = spm.SentencePieceProcessor()
sp.load("./spm_multilang.model")

# 2. Config (same as training)
config = AutoConfig.from_pretrained("google/gemma-3-270m")
config.vocab_size = sp.get_piece_size()
config.hidden_size = 256
config.num_hidden_layers = 4
config.num_attention_heads = 4
config.intermediate_size = 1024
config.max_position_embeddings = 512
config.tie_word_embeddings = True

# 3. Model + checkpoint
model = AutoModelForCausalLM.from_config(config)

ckpt_path = "./data/gemma20m_ckpts/ckpt-83000.pt"
ckpt = torch.load(ckpt_path, map_location="cpu")

if "model" in ckpt:  # training saved {"model": state_dict, ...}
    model.load_state_dict(ckpt["model"])
else:  # in case only raw state_dict
    model.load_state_dict(ckpt)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device).eval()

# 4. Test generation
text = "Hello, how are"
ids = sp.encode(text, out_type=int)
inputs = torch.tensor([ids]).to(device)

with torch.no_grad():
    outputs = model.generate(inputs, max_length=40)

print("Input:", text)
print("Generated:", sp.decode(outputs[0].cpu().tolist()))

# Hindi
text = "नमस्ते, आप कैसे"
ids = sp.encode(text, out_type=int)
inputs = torch.tensor([ids]).to(device)

with torch.no_grad():
    outputs = model.generate(inputs, max_length=40)

print("Input:", text)
print("Generated:", sp.decode(outputs[0].cpu().tolist()))

# Bengali
text = "হ্যালো, আপনি কেমন"  # আছেন
ids = sp.encode(text, out_type=int)
inputs = torch.tensor([ids]).to(device)

with torch.no_grad():
    outputs = model.generate(inputs, max_length=40)

print("Input:", text)
print("Generated:", sp.decode(outputs[0].cpu().tolist()))

Input: Hello, how are
Generated: Hello, how are you going to be, and that you can be able to get the best of your own. The company has also been a part of the new government. The company has also been
Input: नमस्ते, आप कैसे
Generated: नमस्ते, आप कैसे कर सकते हैं? इस दौरान उन्होंने कहा कि इस मामले में पुलिस ने भी इस मामले में जांच की है। इस दौरान उन्होंने कहा कि इस मामले में पुलिस ने भी इस मामले
Input: হ্যালো, আপনি কেমন
Generated: হ্যালো, আপনি কেমন করে?? . . . . . . . . . . . . . . . .


In [ ]:
import torch
from transformers import AutoConfig, AutoModelForCausalLM
import sentencepiece as spm

# 1. Tokenizer
sp = spm.SentencePieceProcessor()
sp.load("./spm_multilang.model")

# 2. Config (same as training)
config = AutoConfig.from_pretrained("google/gemma-3-270m")
config.vocab_size = sp.get_piece_size()
config.hidden_size = 256
config.num_hidden_layers = 4
config.num_attention_heads = 4
config.intermediate_size = 1024
config.max_position_embeddings = 512
config.tie_word_embeddings = True

# 3. Model + checkpoint
model = AutoModelForCausalLM.from_config(config)

ckpt_path = "./data/gemma20m_ckpts/ckpt-95000.pt"
ckpt = torch.load(ckpt_path, map_location="cpu")

if "model" in ckpt:  # training saved {"model": state_dict, ...}
    model.load_state_dict(ckpt["model"])
else:  # in case only raw state_dict
    model.load_state_dict(ckpt)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device).eval()

# 4. Test generation
text = "Hello, how are"
ids = sp.encode(text, out_type=int)
inputs = torch.tensor([ids]).to(device)

with torch.no_grad():
    outputs = model.generate(inputs, max_length=40)

print("Input:", text)
print("Generated:", sp.decode(outputs[0].cpu().tolist()))

# Hindi
text = "नमस्ते, आप कैसे"
ids = sp.encode(text, out_type=int)
inputs = torch.tensor([ids]).to(device)

with torch.no_grad():
    outputs = model.generate(inputs, max_length=40)

print("Input:", text)
print("Generated:", sp.decode(outputs[0].cpu().tolist()))

# Bengali
text = "হ্যালো, আপনি কেমন"  # আছেন
ids = sp.encode(text, out_type=int)
inputs = torch.tensor([ids]).to(device)

with torch.no_grad():
    outputs = model.generate(inputs, max_length=40)

print("Input:", text)
print("Generated:", sp.decode(outputs[0].cpu().tolist()))

Input: Hello, how are
Generated: Hello, how are you going to be, and that's what you're going to be. The company has also been a part of the company’s new and new and new services,
Input: नमस्ते, आप कैसे
Generated: नमस्ते, आप कैसे कर सकते हैं? इस दौरान उन्होंने कहा कि इस मामले में, यह एक महत्वपूर्ण बात है कि यह एक बार फिर से एक बार फिर से एक बार फिर से एक बार फिर
Input: হ্যালো, আপনি কেমন
Generated: হ্যালো, আপনি কেমন করে? আপনি আপনার জন্য আপনার জন্য আপনার আপনার আপনার আপনার আপনার আপনার আপনার আপনার আপনার আপনার আপনার আপনার আপনার আপনার আপনার আপনার আপনার আপনার আপনার আপনার আপনার আপনার আপনার আপনার আপনার আপনার আপনার আপনার
